In [1]:
import shutil
from pathlib import Path

import pandas as pd
from ultralytics import YOLO

from gesture_eval import BASELINE, evaluate, error_analysis

PARAM = "lr0"                      # your hyperparameter: one key of BASELINE (or another Ultralytics setting)
VALUES = [0.001, 0.005, 0.02]      # three values, all different from the baseline's

# 1. Train one model per value (skipped if it is already saved; delete the .pt file to retrain)
for v in VALUES:
    name = f"{PARAM}_{v}"
    weights, out = Path("models") / f"{name}_best.pt", Path("results") / name
    if weights.exists():
        continue
    model = YOLO("yolov8s.pt")
    model.train(**{**BASELINE, PARAM: v}, project="runs", name=name, exist_ok=True)
    run = Path(model.trainer.save_dir)
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(run / "weights" / "best.pt", weights)
    for f in ["results.csv", "results.png", "args.yaml"]:
        shutil.copy2(run / f, out / f)

# 2. Score the baseline and your models with the same code
runs = {BASELINE.get(PARAM, "default"): "baseline", **{v: f"{PARAM}_{v}" for v in VALUES}}
rows = []
for v, name in runs.items():
    weights = Path("models") / f"{name}_best.pt"
    imgsz = v if PARAM == "imgsz" else BASELINE["imgsz"]
    overall, per_class = evaluate(weights, imgsz, Path("results") / name)
    ea = error_analysis(weights, imgsz, Path("results") / name)
    team = overall.iloc[2]         # "test, team only": our own photos
    rows.append({PARAM: v, "run": name, "team mAP50": team["mAP50"], "team mAP50-95": team["mAP50-95"],
                 "team F1": team["F1"], "fist mAP50": per_class.loc[("test (team only)", "fist"), "mAP50"],
                 "F1 @ conf 0.25": ea["summary"]["F1"]})
summary = pd.DataFrame(rows).set_index(PARAM)
summary.round(4).to_csv(Path("results") / f"tuning_{PARAM}_summary.csv")
summary.round(3)

New https://pypi.org/project/ultralytics/8.4.207 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.202  Python-3.14.7 torch-2.11.0+cu128 CPU (AMD Ryzen 7 5825U with Radeon Graphics)


C:\Users\daffo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\cuda\__init__.py:1113: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=lr0_0.001, nbs=64, nms=None, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=15, perspective=0.0, plots=True, pose=12.0, pretrained=True, profi

KeyboardInterrupt: 

In [ ]:
print(pd.DataFrame({s: train_info(ROOT / "results" / f"lr0_{r}" / "lr0results.csv") for r in RATES}).T.rename_axis("lr0"))
fig, axes = plt.subplots(len(RATES), 1, figsize=(16, 8 * len(RATES)))
for ax, s in zip(axes, RATES):
    ax.imshow(plt.imread(ROOT / "results" / f"lr0_{r}" / "results.png")); ax.set_title(f"lr0 {r}"); ax.axis("off")
plt.tight_layout(); plt.show()